# Detectie Deepfake folosind Transfer Learning

In [ ]:
%pip install opencv-python
%pip install matplotlib
%pip install seaborn
%pip install torchvision
%pip install mediapipe
%pip install scikit-learn

In [ ]:
import platform
import sys

print(f"Arhitectură: {platform.architecture()[0]}")
print(f"Versiune Python: {sys.version}")

## 1. Importuri

In [ ]:
# am pus toate importurile intr-un singur loc

# sistem de fisiere
import os
import urllib

# procesare imagini si grafice
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, random_split
from PIL import Image
from torch.utils.data import WeightedRandomSampler

# detectie fata in timp real
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# metrici de evaluare a modelului
from sklearn.metrics import confusion_matrix, classification_report

# verificam de la inceput pe ce rulam - GPU sau CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Ruleaza pe: {device}")

## 2. Extragere Cadre din Video

### 2a. Detectie si Decupare Fata

In [ ]:
# modelul de detectie BlazeFace il descarcam automat daca nu e deja pe disc
TFLITE_MODEL = 'blaze_face_short_range.tflite'

if not os.path.exists(TFLITE_MODEL):
    url = "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite"
    urllib.request.urlretrieve(url, TFLITE_MODEL)
    print("model descarcat")

# initializam detectorul

# MediaPipe foloseste un sistem de 'options' pentru configurare
# min_detection_confidence=0.5 = ignoram detectii cu siguranta sub 50%

_base_opts = python.BaseOptions(model_asset_path=TFLITE_MODEL)
_det_opts  = vision.FaceDetectorOptions(base_options=_base_opts, min_detection_confidence=0.5)
face_detector = vision.FaceDetector.create_from_options(_det_opts)


def crop_face(frame_bgr):
    # OpenCV citeste BGR, MediaPipe vrea RGB - trebuie convertit inainte
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mp_img    = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    results   = face_detector.detect(mp_img)

    # daca nu s-a detectat nicio fata sarim peste frame-ul acesta
    if not results.detections:
        return None

    #  luam prima detectie
    #  bounding box = dreptunghiul care inconjoara fata detectata
    #  origin_x/y = coltul stanga-sus, width/height = dimensiunile 
    bbox = results.detections[0].bounding_box
    h, w = frame_rgb.shape[:2]
    bx, by, bw, bh = bbox.origin_x, bbox.origin_y, bbox.width, bbox.height


    # calculam centrul fetei si aplicam factorul de 1.4 pe latime si inaltime
    cx = bx + bw / 2.0
    cy = by + bh / 2.0
    new_bw = bw * 1.4
    new_bh = bh * 1.4

    # calculam noile colturi folosind centrul
    x1 = max(0, int(cx - new_bw / 2))
    y1 = max(0, int(cy - new_bh / 2))
    x2 = min(w, int(cx + new_bw / 2))
    y2 = min(h, int(cy + new_bh / 2))

    cropped = frame_rgb[y1:y2, x1:x2]

    # verificam sa nu avem un crop gol din cauza coordonatelor invalide
    if cropped.size == 0:
        return None

    # redimensionam la 256x256 - dimensiunea de intrare pentru reteaua MFNN
    return cv2.resize(cropped, (256, 256), interpolation=cv2.INTER_LINEAR)

### 2b. Procesare Videoclipuri -> Imagini

In [ ]:
def video_to_frames(src_dir, dst_dir, fps_target=3):
    # cream folderul de output daca nu exista deja
    os.makedirs(dst_dir, exist_ok=True)

    # construim lista de videoclipuri din directorul sursa
    # f.name[:-4] sterge ultimele 4 caractere (.mp4) din numele fisierului 
    videos = [
        (f.name[:-4], os.path.abspath(f))
        for f in os.scandir(src_dir)
        if f.name.lower().endswith('.mp4')
    ]

    for vid_name, vid_path in videos:
        cap      = cv2.VideoCapture(vid_path)
        orig_fps = cap.get(cv2.CAP_PROP_FPS)

        # calculam la cate cadre citite sa salvam unul
        step = max(1, round(orig_fps / fps_target))

        frame_idx = 0
        saved     = 0

        while True:
            ok, frame = cap.read()
            if not ok:
                break

            # procesam doar cadrele la intervalul calculat, restul le sarim
            if frame_idx % step == 0:
                face = crop_face(frame)
                # salvam doar daca fata a fost detectata cu succes
                if face is not None:

                    fname = f"{vid_name}_{saved:04d}.jpg"
                    # reconvertim la BGR ca OpenCV sa salveze culorile corect
                    cv2.imwrite(os.path.join(dst_dir, fname), cv2.cvtColor(face, cv2.COLOR_RGB2BGR))
                    saved += 1

            frame_idx += 1

        cap.release()
        print(f"{vid_name}: {saved} cadre salvate")


#video_to_frames("dataset_video/fake/manipulated_sequences/DeepFakeDetection/c23/videos", "dataset_image/fake")
#video_to_frames("dataset_video/fake/manipulated_sequences/Face2Face/c23/videos", "dataset_image/fake")
#video_to_frames("dataset_video/fake", "dataset_image/fake")

#video_to_frames("dataset_video/real/original_sequences/actors/c23/videos", "dataset_image/real")
video_to_frames("dataset_video/real/original_sequences/youtube/c23/videos", "dataset_image/real")
#video_to_frames("dataset_video/real", "dataset_image/real")
print("gata extragerea")

## 3. Construirea Modelului (Transfer Learning)

### 3a. Incarcare ResNet18 Preantrenat

In [ ]:
# Definirea arhitecturii MFNN conform articolului

class MFNN(nn.Module):
    def __init__(self, num_classes=2):
        super(MFNN, self).__init__()
        # Folosim un ResNet18 ca model de baza
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        # Nivel 1: Primele straturi (micro-artefacte/zgomot statistic)
        self.shallow_block = nn.Sequential(*list(backbone.children())[:5]) 

        # Nivel 2: Straturile de mijloc (trasaturi mezoscopice/texturi)
        self.middle_block = backbone.layer2 
        
        # Nivel 3: Straturile profunde (semantica fetei/coerenta)
        self.deep_block = nn.Sequential(backbone.layer3, backbone.layer4)
        
        # Straturi de adaptare pentru fuziune (sa aducem totul la aceeasi dimensiune)
        # Articolul zice ca le aducem la un numitor comun inainte de concatenare
        self.conv_shallow = nn.Conv2d(64, 256, kernel_size=8, stride=8)
        self.conv_middle = nn.Conv2d(128, 256, kernel_size=4, stride=4)
        self.conv_deep = nn.Conv2d(512, 256, kernel_size=1, stride=1)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        # Clasificatorul final: primeste 256 * 3 trasaturi fuzionate
        self.classifier = nn.Linear(256 * 3, num_classes)

    def forward(self, x):
        f1 = self.shallow_block(x)       # Iesire Shallow
        f2 = self.middle_block(f1)       # Iesire Middle
        f3 = self.deep_block(f2)         # Iesire Deep
        
        # Procesul de fuziune
        t1 = self.conv_shallow(f1)
        t2 = self.conv_middle(f2)
        t3 = self.conv_deep(f3)
        
        combined = torch.cat((t1, t2, t3), dim=1)
        out = self.avgpool(combined)
        out = torch.flatten(out, 1)
        return self.classifier(out)

# Instantierea modelului MFNN
model = MFNN(num_classes=2).to(device)
print("Model MFNN initializat (S-M-D Fusion)")

In [ ]:
# afisam arhitectura
print(model)

## 4. Pregatire Dataset si DataLoaders

In [ ]:
# definim transformarile pentru antrenament cu augmentare

train_transforms = transforms.Compose([

    transforms.Resize((256, 256)),

    # flip orizontal - simuleaza fete orientate in directii diferite
    transforms.RandomHorizontalFlip(),

    # rotatie mica - simuleaza capete usor inclinate in video
    transforms.RandomRotation(15),

    # variatie de luminozitate si contrast - camere si iluminat diferit
    transforms.ColorJitter(brightness=0.2, contrast=0.2),

    # convertim imaginea PIL intr-un tensor PyTorch si scalat in [0, 1]
    transforms.ToTensor(),
    
    # normalizare cu valorile standard de pe ImageNet - necesar pentru modelul preantrenat
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from sklearn.model_selection import train_test_split

# citesc pozele din foldere. ImageFolder stie singur care e clasa 0 si care e 1
full_dataset = datasets.ImageFolder(root="dataset_image", transform=train_transforms)
print("Clase:", full_dataset.class_to_idx)

# extrag intai toate etichetele ca sa stiu exact ce proportii am
all_targets = full_dataset.targets

# folosesc train_test_split din sklearn cu "stratify".
# asta ma asigura ca pastreaza fix aceeasi proportie de Fake/Real si la train si la test
train_idx, test_idx = train_test_split(
    range(len(full_dataset)),
    test_size=0.2,
    random_state=42,
    stratify=all_targets
)

# construiesc noile seturi folosind indicii extrasi mai sus
train_set = Subset(full_dataset, train_idx)
test_set = Subset(full_dataset, test_idx)

print(f"Total imagini train: {len(train_set)} | Total imagini test: {len(test_set)}")


# ECHILIBRAREA CLASELOR 


# numar de cate ori apare fiecare clasa in noul set de train
labels_train = [all_targets[i] for i in train_idx]
class_counts = np.bincount(labels_train)
print(f"Avem la train -> Fake: {class_counts[0]} | Real: {class_counts[1]}")

# ii dau o greutate mult mai mare clasei Real (ca sunt putine) fata de Fake (care sunt vreo 148k)
class_weights = 1.0 / class_counts
sample_weights = [class_weights[lbl] for lbl in labels_train]
sample_weights = torch.DoubleTensor(sample_weights)

# face ca la fiecare epoca, imaginile sa fie trase in functie de greutatea de mai sus
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# pun in loadere (doar la train pun sampler-ul pt ca acolo invata, la test le las normale)
train_loader = DataLoader(train_set, batch_size=32, sampler=sampler, num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=32, shuffle=False, num_workers=0)

## 5. Antrenare

In [ ]:
criterion = nn.CrossEntropyLoss()


optimizer = optim.Adam(model.parameters(), lr=1e-4)


NUM_EPOCHS = 1

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    correct    = 0
    total      = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        
        # Imaginile trec prin MFNN, fac fuziunea pe cele 3 straturi
        # outputs va avea forma [batch_size, 2] la fel ca inainte
        outputs = model(images) 
        loss    = criterion(outputs, labels)

        # Backpropagation - acum va actualiza si straturile custom
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        
        # torch.max returneaza (valori_maxime, indici_maximi)
        # Indicele (0 sau 1) reprezinta clasa prezisa
        _, predicted = torch.max(outputs, 1)
        
        correct += (predicted == labels).sum().item()
        total   += labels.size(0)

    avg_loss = total_loss / total
    acc      = correct / total * 100
    print(f"Epoca {epoch+1}/{NUM_EPOCHS} | loss: {avg_loss:.4f} | acc: {acc:.2f}%")

print("antrenare terminata")

In [ ]:
# salvam doar greutatile modelului, nu intreaga structura
SAVE_PATH = "model_mfnn_deepfake4.pth"
torch.save(model.state_dict(), SAVE_PATH)
print(f"model salvat: {SAVE_PATH}")

## 6. Evaluare pe Setul de Test

In [ ]:
import copy
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image
import io
import matplotlib.pyplot as plt
import seaborn as sns

# setarile pt pozele normale de test
normal_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class LowQualityJPEG(object):
    def __init__(self, quality=15):
        self.quality = quality
        
    def __call__(self, img):
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=self.quality)
        buffer.seek(0) # ma intorc la inceputul fisierului din memorie
        return Image.open(buffer)


stress_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    LowQualityJPEG(quality=12), # face poza super pixelata
    transforms.ColorJitter(brightness=(0.3, 0.5)), # o face intunecata
    transforms.ToTensor(),
    transforms.RandomErasing(p=1.0, scale=(0.05, 0.15), value='random'), # baga zgomot random pe fata
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


normal_dataset = copy.deepcopy(full_dataset)
normal_dataset.transform = normal_transforms
normal_test_set = Subset(normal_dataset, test_set.indices) 
normal_loader = DataLoader(normal_test_set, batch_size=32, shuffle=False)

stress_dataset = copy.deepcopy(full_dataset)
stress_dataset.transform = stress_transforms
stress_test_set = Subset(stress_dataset, test_set.indices) 
stress_loader = DataLoader(stress_test_set, batch_size=32, shuffle=False)

def evaluate_model(eval_model, data_loader, test_name="Test"):
    eval_model.eval() 
    all_labels = []
    all_preds = []
    correct = 0
    total = 0

    print(f"\nSe testeaza: {test_name} ")

    with torch.no_grad(): 
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = eval_model(images)
        
            _, preds = torch.max(outputs, 1)
            
            total += labels.size(0)
            correct += (preds == labels).sum().item()
            
    
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    accuracy = (correct / total) * 100
    print(f"Cat a ghicit: {accuracy:.2f}%")
    
  
    print(classification_report(all_labels, all_preds, target_names=['Fake', 'Real']))

   
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Pred Fake', 'Pred Real'],
                yticklabels=['True Fake', 'True Real'])
    plt.title(f'Matrice - {test_name}')
    plt.show()

    return accuracy



normal_accuracy = evaluate_model(model, normal_loader, test_name="Poze Bune")


stress_accuracy = evaluate_model(model, stress_loader, test_name="Stress Test")

print("\n--- Concluzii ---")
print(f"Acuratete poze clare: {normal_accuracy:.2f}%")
print(f"Acuratete poze alterate: {stress_accuracy:.2f}%")
print(f"Scadere totala: {normal_accuracy - stress_accuracy:.2f} %")

## 7. Interfata Grafica pentru Testare

In [ ]:
import os
import sys
import threading
import time
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image, ImageTk
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


dispozitiv = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nRuntime: {dispozitiv.type.upper()}")

class MFNN(nn.Module):
    def __init__(self, num_clase=2):
        super(MFNN, self).__init__()
        
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        self.shallow_block = nn.Sequential(*list(backbone.children())[:5])
        self.middle_block = backbone.layer2
        self.deep_block = nn.Sequential(backbone.layer3, backbone.layer4)
        
        self.conv_shallow = nn.Conv2d(64, 256, kernel_size=8, stride=8)
        self.conv_middle = nn.Conv2d(128, 256, kernel_size=4, stride=4)
        self.conv_deep = nn.Conv2d(512, 256, kernel_size=1, stride=1)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(256 * 3, num_clase)

    def forward(self, x):
        f_shallow = self.shallow_block(x)
        f_middle = self.middle_block(f_shallow)
        f_deep = self.deep_block(f_middle)
        
        t_s = self.conv_shallow(f_shallow)
        t_m = self.conv_middle(f_middle)
        t_d = self.conv_deep(f_deep)
        
        features_fused = torch.cat((t_s, t_m, t_d), dim=1)
        pooled = self.avgpool(features_fused)
        flattened = torch.flatten(pooled, 1)
        
        return self.classifier(flattened)


MODEL_FATA = 'blaze_face_short_range.tflite'

if not os.path.exists(MODEL_FATA):
    import urllib.request
    print("Se descarca detector BlazeFace...")
    url = "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite"
    urllib.request.urlretrieve(url, MODEL_FATA)
    print("Downloaded.\n")

opts_blazeface = python.BaseOptions(model_asset_path=MODEL_FATA)
opts_detector = vision.FaceDetectorOptions(
    base_options=opts_blazeface, 
    min_detection_confidence=0.5
)
detectorul_fata = vision.FaceDetector.create_from_options(opts_detector)


def extrage_fata(cadru_bgr):
    cadru_rgb = cv2.cvtColor(cadru_bgr, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=cadru_rgb)
    rezultate = detectorul_fata.detect(mp_img)
    
    if not rezultate.detections:
        return None, None
    
    bbox = rezultate.detections[0].bounding_box
    h, w = cadru_rgb.shape[:2]
    
    origin_x = bbox.origin_x
    origin_y = bbox.origin_y
    box_w = bbox.width
    box_h = bbox.height
    
    centru_x = origin_x + box_w / 2.0
    centru_y = origin_y + box_h / 2.0
    
    nou_w = box_w * 1.4
    nou_h = box_h * 1.4
    
    x1 = max(0, int(centru_x - nou_w / 2))
    y1 = max(0, int(centru_y - nou_h / 2))
    x2 = min(w, int(centru_x + nou_w / 2))
    y2 = min(h, int(centru_y + nou_h / 2))
    
    bbox_coords = (x1, y1, x2, y2)
    decupata = cadru_rgb[y1:y2, x1:x2]
    
    if decupata.size == 0:
        return None, None
    
    redimensionata = cv2.resize(decupata, (256, 256), interpolation=cv2.INTER_LINEAR)
    return redimensionata, bbox_coords


transformare_normal = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


def incarca_model():
    print("Se incarca modelul MFNN...")
    model_instanta = MFNN(num_clase=2).to(dispozitiv)
    
    cai_posibile = [
        "model_mfnn_deepfake_2.pth",
        "model_mfnn_deepfake_2",
        "./model_mfnn_deepfake_2.pth",
    ]
    
    incarcat = False
    for cale_model in cai_posibile:
        try:
            if os.path.exists(cale_model):
                print(f"Model gasit: {cale_model}")
                state_dict = torch.load(cale_model, map_location=dispozitiv)
                model_instanta.load_state_dict(state_dict)
                num_params = sum(p.numel() for p in model_instanta.parameters())
                print(f"Parametri: {num_params:,}")
                incarcat = True
                break
        except Exception as e:
            continue
    
    if not incarcat:
        print("\nATENTIE: Greutatile modelului nu au fost gasite!")
        print("Rulare cu initializare random (rezultate nesigure)")
    
    model_instanta.eval()
    return model_instanta


model_deepfake = incarca_model()


class DeepfakeDetectorUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Detector DeepFake v2.1")
        self.root.geometry("1450x920")
        self.root.configure(bg="#0f1419")
        
        self.video_curent = None
        self.se_proceseaza = False
        self.semnal_stop = False
        self.predicti_istorie = []
        self.confidence_istorie = []
        self.nr_cadre_procesat = []
        
        self.culori = {
            'bg_dark': '#0f1419',
            'bg_panel': '#1a1f2e',
            'accent_cyan': '#00d9ff',
            'accent_green': '#2ecc71',
            'accent_red': '#e74c3c',
            'accent_yellow': '#f39c12',
            'text_main': '#ecf0f1',
            'text_secondary': '#bdc3c7'
        }
        
        self.root.protocol("WM_DELETE_WINDOW", self.inchide_curat)
        self._construieste_ui()
        print("\nInterfata initializata\n")

    def _construieste_ui(self):
        header_frame = tk.Frame(self.root, bg=self.culori['accent_cyan'], height=70)
        header_frame.pack(fill=tk.X)
        header_frame.pack_propagate(False)
        
        tk.Label(
            header_frame,
            text="Sistem Detectie DeepFake",
            font=("Courier New", 24, "bold"),
            bg=self.culori['accent_cyan'],
            fg="#000000"
        ).pack(pady=15)
    
        
        container = tk.Frame(self.root, bg=self.culori['bg_dark'])
        container.pack(fill=tk.BOTH, expand=True, padx=8, pady=8)
        
        left_panel = tk.Frame(
            container,
            bg=self.culori['bg_panel'],
            highlightbackground=self.culori['accent_cyan'],
            highlightthickness=2,
            width=380
        )
        left_panel.pack(side=tk.LEFT, fill=tk.Y, padx=5, pady=5)
        left_panel.pack_propagate(False)
        
        tk.Label(
            left_panel,
            text="INPUT VIDEO",
            font=("Monaco", 13, "bold"),
            bg=self.culori['bg_panel'],
            fg=self.culori['accent_cyan']
        ).pack(pady=(15, 10))
        
        self.btn_select = tk.Button(
            left_panel,
            text="Selecteaza Fisier Video",
            command=self.selecteaza_video,
            font=("Monaco", 11, "bold"),
            bg=self.culori['accent_green'],
            fg="#000000",
            padx=15, pady=12,
            activebackground="#27ae60",
            relief=tk.FLAT,
            cursor="hand2"
        )
        self.btn_select.pack(pady=8, padx=15, fill=tk.X)
        
        self.label_cale_video = tk.Label(
            left_panel,
            text="[ Niciun fisier selectat ]",
            font=("Monaco", 9),
            bg=self.culori['bg_panel'],
            fg=self.culori['text_secondary'],
            wraplength=330,
            justify=tk.LEFT
        )
        self.label_cale_video.pack(pady=8, padx=12)
        
        ttk.Separator(left_panel, orient=tk.HORIZONTAL).pack(
            fill=tk.X, pady=15, padx=12
        )
        
        tk.Label(
            left_panel,
            text="MODURI ANALIZA",
            font=("Monaco", 13, "bold"),
            bg=self.culori['bg_panel'],
            fg=self.culori['accent_cyan']
        ).pack(pady=(10, 10))
        
        self.btn_test_normal = tk.Button(
            left_panel,
            text="CLEAN TEST",
            command=self.incepe_test_normal,
            font=("Monaco", 10, "bold"),
            bg="#27ae60",
            fg="white",
            padx=15, pady=11,
            activebackground="#229954",
            relief=tk.FLAT,
            cursor="hand2",
            state=tk.DISABLED
        )
        self.btn_test_normal.pack(pady=7, padx=15, fill=tk.X)
        
        # --- Sectiunea STRESS TEST ---
        frame_stress = tk.Frame(left_panel, bg=self.culori['bg_panel'])
        frame_stress.pack(pady=7, padx=15, fill=tk.X)
        
        self.var_tip_stress = tk.StringVar(value="Compresie JPEG")
        self.combo_stress = ttk.Combobox(
            frame_stress, 
            textvariable=self.var_tip_stress,
            values=["Compresie JPEG", "Blur", "Artefacte/Pixeli", "Combinat (Extrem)"],
            state="readonly",
            font=("Monaco", 10)
        )
        self.combo_stress.pack(side=tk.TOP, fill=tk.X, pady=(0, 5))
        
        self.btn_test_stress = tk.Button(
            frame_stress,
            text="STRESS TEST LIVE",
            command=self.incepe_test_stress,
            font=("Monaco", 10, "bold"),
            bg="#e74c3c",
            fg="white",
            padx=15, pady=11,
            activebackground="#c0392b",
            relief=tk.FLAT,
            cursor="hand2",
            state=tk.DISABLED
        )
        self.btn_test_stress.pack(side=tk.TOP, fill=tk.X)

        self.btn_stop = tk.Button(
            left_panel,
            text="STOP",
            command=self.opreste_procesare,
            font=("Monaco", 10, "bold"),
            bg="#7f8c8d",
            fg="white",
            padx=15, pady=11,
            activebackground="#5d6d7b",
            relief=tk.FLAT,
            cursor="hand2",
            state=tk.DISABLED
        )
        self.btn_stop.pack(pady=15, padx=15, fill=tk.X)
        
        ttk.Separator(left_panel, orient=tk.HORIZONTAL).pack(
            fill=tk.X, pady=15, padx=12
        )
        
        tk.Label(
            left_panel,
            text="STATUS",
            font=("Monaco", 13, "bold"),
            bg=self.culori['bg_panel'],
            fg=self.culori['accent_cyan']
        ).pack(pady=(10, 8))
        
        self.label_status = tk.Label(
            left_panel,
            text="Pregatit.",
            font=("Monaco", 9, "italic"),
            bg=self.culori['bg_panel'],
            fg=self.culori['accent_yellow'],
            wraplength=330,
            justify=tk.LEFT
        )
        self.label_status.pack(pady=8, padx=12)
        
        self.progress_bar = ttk.Progressbar(
            left_panel,
            length=300,
            mode='indeterminate',
            style='TProgressbar'
        )
        self.progress_bar.pack(pady=10, padx=15, fill=tk.X)
        
        center_panel = tk.Frame(
            container,
            bg=self.culori['bg_panel'],
            highlightbackground=self.culori['accent_cyan'],
            highlightthickness=2
        )
        center_panel.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=5, pady=5)
        
        tk.Label(
            center_panel,
            text="LIVE PREVIEW",
            font=("Monaco", 13, "bold"),
            bg=self.culori['bg_panel'],
            fg=self.culori['accent_cyan']
        ).pack(pady=10)
        
        self.canvas_video = tk.Canvas(
            center_panel,
            bg="#000000",
            width=640,
            height=480,
            highlightbackground=self.culori['accent_cyan'],
            highlightthickness=1
        )
        self.canvas_video.pack(pady=8, padx=8, fill=tk.BOTH, expand=True)
        
        self.label_cadru_info = tk.Label(
            center_panel,
            text="Se asteapta input video...",
            font=("Monaco", 9),
            bg=self.culori['bg_panel'],
            fg=self.culori['accent_green']
        )
        self.label_cadru_info.pack(pady=5)
        
        right_panel = tk.Frame(
            container,
            bg=self.culori['bg_panel'],
            highlightbackground=self.culori['accent_cyan'],
            highlightthickness=2,
            width=340
        )
        right_panel.pack(side=tk.RIGHT, fill=tk.BOTH, expand=False, padx=5, pady=5)
        right_panel.pack_propagate(False)
        
        tk.Label(
            right_panel,
            text="VERDICT FINAL",
            font=("Monaco", 13, "bold"),
            bg=self.culori['bg_panel'],
            fg=self.culori['accent_cyan']
        ).pack(pady=12)
        
        self.label_verdict = tk.Label(
            right_panel,
            text="-" * 30,
            font=("Monaco", 18, "bold"),
            bg=self.culori['bg_panel'],
            fg=self.culori['text_main']
        )
        self.label_verdict.pack(pady=15)
        
        self.label_raport = tk.Label(
            right_panel,
            text="In asteptarea analizei...",
            font=("Courier", 9),
            bg=self.culori['bg_panel'],
            fg=self.culori['accent_green'],
            justify=tk.LEFT,
            wraplength=310
        )
        self.label_raport.pack(pady=12, padx=10)
        
        ttk.Separator(right_panel, orient=tk.HORIZONTAL).pack(
            fill=tk.X, pady=12, padx=10
        )
        
        self.fig_confidence = Figure(
            figsize=(3.5, 2.8),
            dpi=85,
            facecolor=self.culori['bg_panel']
        )
        self.ax_confidence = self.fig_confidence.add_subplot(111, facecolor=self.culori['bg_panel'])
        self.ax_confidence.set_facecolor(self.culori['bg_panel'])
        self.ax_confidence.tick_params(colors=self.culori['text_secondary'], labelsize=8)
        
        for spine in ['bottom', 'left']:
            self.ax_confidence.spines[spine].set_color(self.culori['text_secondary'])
        for spine in ['top', 'right']:
            self.ax_confidence.spines[spine].set_visible(False)
        
        self.canvas_graph = FigureCanvasTkAgg(self.fig_confidence, master=right_panel)
        self.canvas_graph.get_tk_widget().pack(fill=tk.BOTH, expand=True, padx=8, pady=8)
        
        self._refresh_graph()

    def _refresh_graph(self):
        self.ax_confidence.clear()
        
        if len(self.confidence_istorie) > 0:
            culori_bare = [
                self.culori['accent_red'] if p == 0 else self.culori['accent_green']
                for p in self.predicti_istorie
            ]
            self.ax_confidence.bar(
                range(len(self.confidence_istorie)),
                self.confidence_istorie,
                color=culori_bare,
                alpha=0.8,
                width=0.7
            )
            self.ax_confidence.set_ylabel('Confidence %', color=self.culori['text_secondary'], fontsize=9)
            self.ax_confidence.set_xlabel('Frames', color=self.culori['text_secondary'], fontsize=9)
            self.ax_confidence.set_ylim([0, 105])
            self.ax_confidence.grid(True, alpha=0.15, linestyle='--')
        else:
            self.ax_confidence.text(
                0.5, 0.5,
                'Fara date',
                ha='center', va='center',
                transform=self.ax_confidence.transAxes,
                color=self.culori['accent_cyan'],
                fontsize=11,
                style='italic'
            )
        
        self.ax_confidence.tick_params(colors=self.culori['text_secondary'])
        self.fig_confidence.tight_layout()
        self.canvas_graph.draw_idle()

    def selecteaza_video(self):
        cale = filedialog.askopenfilename(
            title="Selecteaza fisier video",
            filetypes=[("Video", "*.mp4 *.avi *.mov *.mkv"), ("Toate fisierele", "*.*")]
        )
        
        if cale:
            self.video_curent = cale
            self.label_cale_video.config(
                text=f"{os.path.basename(cale)}",
                fg=self.culori['accent_green']
            )
            self.btn_test_normal.config(state=tk.NORMAL)
            self.btn_test_stress.config(state=tk.NORMAL)
            self.label_status.config(
                text="Fisier incarcat. Pregatit pentru analiza.",
                fg=self.culori['accent_green']
            )

    def aplica_degradare(self, cadru_rgb, tip):
        """Aplica artefacte vizuale direct pe cadrul video"""
        if tip == "Compresie JPEG":
            encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), 8]
            bgr = cv2.cvtColor(cadru_rgb, cv2.COLOR_RGB2BGR)
            _, encimg = cv2.imencode('.jpg', bgr, encode_param)
            decimg = cv2.imdecode(encimg, 1)
            return cv2.cvtColor(decimg, cv2.COLOR_BGR2RGB)
            
        elif tip == "Blur":
            return cv2.GaussianBlur(cadru_rgb, (15, 15), 0)
            
        elif tip == "Artefacte/Pixeli":
            h, w = cadru_rgb.shape[:2]
            mic = cv2.resize(cadru_rgb, (w // 10, h // 10), interpolation=cv2.INTER_LINEAR)
            return cv2.resize(mic, (w, h), interpolation=cv2.INTER_NEAREST)
            
        elif tip == "Combinat (Extrem)":
            h, w = cadru_rgb.shape[:2]
            # Pixelare
            mic = cv2.resize(cadru_rgb, (w // 8, h // 8), interpolation=cv2.INTER_LINEAR)
            pix = cv2.resize(mic, (w, h), interpolation=cv2.INTER_NEAREST)
            # Compresie pe deasupra
            bgr = cv2.cvtColor(pix, cv2.COLOR_RGB2BGR)
            encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), 6]
            _, encimg = cv2.imencode('.jpg', bgr, encode_param)
            decimg = cv2.imdecode(encimg, 1)
            return cv2.cvtColor(decimg, cv2.COLOR_BGR2RGB)
            
        return cadru_rgb

    def incepe_test_normal(self):
        if not self.video_curent:
            messagebox.showerror("Eroare", "Selecteaza un fisier video mai intai!")
            return
        
        self.semnal_stop = False
        self.se_proceseaza = True
        self._reset_analysis()
        
        self.btn_test_normal.config(state=tk.DISABLED)
        self.btn_test_stress.config(state=tk.DISABLED)
        self.combo_stress.config(state=tk.DISABLED)
        self.btn_stop.config(state=tk.NORMAL)
        
        self.label_status.config(
            text="Se ruleaza testul CLEAN...",
            fg=self.culori['accent_cyan']
        )
        self.progress_bar.start()
        
        thread = threading.Thread(
            target=self._proceseaza_video,
            args=(self.video_curent, False),
            daemon=True
        )
        thread.start()

    def incepe_test_stress(self):
        if not self.video_curent:
            messagebox.showerror("Eroare", "Selecteaza un fisier video mai intai!")
            return
        
        self.semnal_stop = False
        self.se_proceseaza = True
        self._reset_analysis()
        
        self.btn_test_normal.config(state=tk.DISABLED)
        self.btn_test_stress.config(state=tk.DISABLED)
        self.combo_stress.config(state=tk.DISABLED)
        self.btn_stop.config(state=tk.NORMAL)
        
        tip = self.var_tip_stress.get()
        self.label_status.config(
            text=f"Se ruleaza STRESS ({tip})...",
            fg=self.culori['accent_red']
        )
        self.progress_bar.start()
        
        thread = threading.Thread(
            target=self._proceseaza_video,
            args=(self.video_curent, True),
            daemon=True
        )
        thread.start()

    def opreste_procesare(self):
        self.semnal_stop = True
        self.label_status.config(
            text="Se opreste...",
            fg=self.culori['accent_yellow']
        )

    def _reset_analysis(self):
        self.predicti_istorie = []
        self.confidence_istorie = []
        self.nr_cadre_procesat = []
        self.label_verdict.config(
            text="SE ANALIZEAZA...",
            fg=self.culori['accent_cyan']
        )
        self.label_raport.config(text="Se proceseaza video...", fg=self.culori['accent_cyan'])
        self._refresh_graph()

    def _proceseaza_video(self, cale_video, este_stress=False):
        try:
            captura = cv2.VideoCapture(cale_video)
            if not captura.isOpened():
                self.label_status.config(
                    text="Nu se poate deschide fisierul video!",
                    fg=self.culori['accent_red']
                )
                return
            
            voturi_fake = 0
            voturi_real = 0
            nr_cadru = 0
            idx_cadru = 0
            
            tip_stress = self.var_tip_stress.get()
            
            with torch.no_grad():
                while True:
                    if self.semnal_stop:
                        break
                    
                    ok, cadru = captura.read()
                    if not ok:
                        break
                    
                    idx_cadru += 1
                    if idx_cadru % 5 != 0:
                        continue
                    
                    nr_cadru += 1
                    
                    
                    fata_decupata_curata, bbox = extrage_fata(cadru)
                    if fata_decupata_curata is None:
                        continue
                    
                    cadru_display = cv2.cvtColor(cadru, cv2.COLOR_BGR2RGB)
                    
                    
                    if este_stress:
                        cadru_display = self.aplica_degradare(cadru_display, tip_stress)
                        
                       
                        x1, y1, x2, y2 = bbox
                        h, w = cadru_display.shape[:2]
                        x1, y1 = max(0, x1), max(0, y1)
                        x2, y2 = min(w, x2), min(h, y2)
                        
                        zona_degradata = cadru_display[y1:y2, x1:x2]
                        if zona_degradata.size > 0:
                            fata_pentru_model = cv2.resize(zona_degradata, (256, 256), interpolation=cv2.INTER_LINEAR)
                        else:
                            fata_pentru_model = fata_decupata_curata
                    else:
                        fata_pentru_model = fata_decupata_curata
                    
                    if bbox:
                        x1, y1, x2, y2 = bbox
                        cv2.rectangle(cadru_display, (x1, y1), (x2, y2), (0, 255, 0), 3)
                        cv2.putText(
                            cadru_display,
                            f"Frame {nr_cadru}",
                            (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.9, (0, 255, 0), 2
                        )
                    
                    cadru_redimensionat = cv2.resize(cadru_display, (640, 480))
                    img_pil = Image.fromarray(cadru_redimensionat)
                    img_tk = ImageTk.PhotoImage(img_pil)
                    
                    self.canvas_video.create_image(0, 0, anchor=tk.NW, image=img_tk)
                    self.canvas_video.image = img_tk
                    
                   
                    tensor = transformare_normal(Image.fromarray(fata_pentru_model))
                    tensor = tensor.unsqueeze(0).to(dispozitiv)
                    
                    output = model_deepfake(tensor)
                    probabilitati = F.softmax(output, dim=1)[0]
                    predicatie = torch.argmax(probabilitati).item()
                    confidence_score = (probabilitati[predicatie] * 100).item()
                    
                    self.predicti_istorie.append(predicatie)
                    self.confidence_istorie.append(confidence_score)
                    self.nr_cadre_procesat.append(nr_cadru)
                    
                    if predicatie == 0:
                        voturi_fake += 1
                        eticheta = "FAKE"
                    else:
                        voturi_real += 1
                        eticheta = "REAL"
                    
                    msg_log = f"Frame {nr_cadru:3d} | {eticheta} ({confidence_score:5.1f}%)"
                    
                    self.label_cadru_info.config(text=msg_log)
                    self._refresh_graph()
                    self.root.update_idletasks()
                    
                    if nr_cadru >= 100:
                        break
            
            captura.release()
            self._finalizare_analiza(voturi_fake, voturi_real, este_stress)
        
        except Exception as err:
            self.label_status.config(
                text=f"Eroare: {str(err)[:40]}...",
                fg=self.culori['accent_red']
            )
        
        finally:
            self.progress_bar.stop()
            self.se_proceseaza = False
            self.btn_test_normal.config(state=tk.NORMAL)
            self.btn_test_stress.config(state=tk.NORMAL)
            self.combo_stress.config(state="readonly")
            self.btn_stop.config(state=tk.DISABLED)

    def _finalizare_analiza(self, voturi_fake, voturi_real, este_stress):
        total = voturi_fake + voturi_real
        
        if total == 0:
            self.label_verdict.config(text="FARA FETE", fg=self.culori['accent_red'])
            self.label_raport.config(text="Nu au fost detectate fete in video.", fg=self.culori['accent_red'])
            self.label_status.config(text="Analiza a esuat.", fg=self.culori['accent_red'])
            return
        
        procent_fake = (voturi_fake / total) * 100
        procent_real = (voturi_real / total) * 100
        
        if voturi_fake > voturi_real:
            verdict = "DEEPFAKE"
            culoare = self.culori['accent_red']
            status_msg = "Continut sintetic detectat!"
        else:
            verdict = "AUTENTIC"
            culoare = self.culori['accent_green']
            status_msg = "Continutul pare autentic."
        
        detalii_mod = self.var_tip_stress.get() if este_stress else "CLEAN"
        raport_text = f"Detectii Fake: {voturi_fake}\nDetectii Real: {voturi_real}\nTotal cadre procesate: {total}\n\nScor Deepfake: {procent_fake:.1f}%\nScor Autentic: {procent_real:.1f}%\n\nMod: {detalii_mod}"
        
        self.label_verdict.config(text=verdict, fg=culoare)
        self.label_raport.config(text=raport_text.strip(), fg=self.culori['accent_green'])
        self.label_status.config(text=status_msg, fg=culoare)

    def inchide_curat(self):
        if self.se_proceseaza:
            self.semnal_stop = True
            time.sleep(0.5)
        
        self.root.quit()
        self.root.destroy()


if __name__ == "__main__":
    fereastra_principala = tk.Tk()
    aplicatia = DeepfakeDetectorUI(fereastra_principala)
    fereastra_principala.mainloop()


Runtime: CUDA
Se incarca modelul MFNN...
Model gasit: model_mfnn_deepfake_2.pth
Parametri: 12,882,754

Interfata initializata

